In [21]:
### IMPORT EXCEPTION MODULES
from requests.exceptions import Timeout
from github import GithubException, UnknownObjectException, IncompletableObject

### IMPORT SYSTEM MODULES
from github import Github
import os, logging, pandas, csv, tempfile, shutil
pandas.set_option('future.no_silent_downcasting', True)

from datetime import datetime, timezone, timedelta
from tqdm import tqdm
from pathlib import Path
import numpy as np

from truckfactor.compute import main as compute_tf
import re
import unicodedata
from typing import Iterable, Tuple, List, Set, Dict, List, Optional, Tuple
from collections import Counter

### IMPORT CUSTOM MODULES
import sys
sys.path.append('../')
import Settings as cfg
import Utilities as util
import subprocess, tempfile, shutil

from git import Repo, exc as git_exc
import time

import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

from dataclasses import dataclass
from typing import Optional, List


def view_df(df, name="DataFrame"):
    import tempfile, webbrowser
    html = "\n".join([
        "<meta charset='utf-8'>",
        "<style>body{font-family:system-ui,Segoe UI,Arial}table{border-collapse:collapse}th,td{border:1px solid #ddd;padding:6px}th{position:sticky;top:0;background:#fafafa}</style>",
        f"<h3>{name}</h3>",
        df.to_html(index=False, escape=False),
    ])
    with tempfile.NamedTemporaryFile("w", delete=False, suffix=".html", encoding="utf-8") as f:
        f.write(html)
        webbrowser.open("file://" + f.name)

# NEW BREAK IDENTIFICATION 
                                         (MADE BY SAM UTZ)

In [22]:
def get_NONCODING(folder: str, dev_login: str) -> pandas.DataFrame:

    """
    Build the developer's DAILY 'other-actions' table.
    Returns a dataframe whose index is the *action*
    ('issues/pull_requests', 'issues_comments', …) and whose
    columns are day-strings.
    """
    files = {
    "prs": (
        "prs_repo.csv",
        {"PR_id": "id", "created_at": "date", "created_by": "creator_login"},
    ),
    "prs_comments": (
        "prs_comments.csv",
        {"comment_id": "id", "created_at": "date", "created_by": "creator_login"},
    ),
    "issues": (
        "issues_repo.csv",
        {"issue_id": "id", "created_at": "date", "created_by": "creator_login"},
    ),
    "issues_comments": (
        "issues_comments_repo.csv",
        {"comment_id": "id", "created_at": "date", "created_by": "creator_login"},
    ),
    "issues_events": (
        "issues_events_repo.csv",
        {"event_id": "id", "created_at": "date", "created_by": "creator_login"},
    ),
    "issues_timeline": (
        "issues_timeline_repo.csv",
        {"event_id": "id", "created_at": "date", "created_by": "creator_login"},
    )
    }

    # ---------- read / filter every file ----------
    dfs = {}
    for key, (fname, rename_map) in files.items():
        dfs[key] = _load_activity_csv(folder, fname, rename_map, dev_login)

    # ---------- split issues vs PRs -------------
    # Old logic: issues endpoint also returns PRs; remove rows whose id
    # matches a PR id so we don’t double-count.
    if not dfs["issues"].empty and not dfs["prs"].empty:
        dfs["issues"] = dfs["issues"][~dfs["issues"].id.isin(dfs["prs"].id)]

    # ---------- build the day range -------------
    # Derive it from the *actual* activity we just read.
    #
    # 1) gather every non-empty dataframe
    non_empty = [df for df in dfs.values() if not df.empty]

    if non_empty:
        # 2) earliest / latest date across *all* action types
        min_date = min(df["date"].min() for df in non_empty)
        max_date = max(df["date"].max() for df in non_empty)
    else:
        # Developer has no activity at all → default to one-day range
        min_date = max_date = pandas.Timestamp.today()

    # 3) full, dense list of day strings
    day_cols = (
        pandas.date_range(
            start=pandas.to_datetime(min_date).normalize(),
            end=pandas.to_datetime(max_date).normalize(),
            freq="D",
        )
        .strftime("%Y-%m-%d")
        .tolist()
    )

    # ---------- helper to create one timeline row ----------
    def _timeline_row(action_name, df_raw):
        row = [action_name]
        if df_raw.empty:
            row += [0] * len(day_cols)
            return row
        counts = (
            pandas.to_datetime(df_raw["date"])
            .dt.date
            .value_counts()
            .to_dict()
        )
        for d in day_cols:
            row.append(counts.get(pandas.to_datetime(d).date(), 0))
        return row

    # ---------- compile all action rows ----------
    rows = []
    if not dfs["issues"].empty:
        rows.append(_timeline_row("issues", dfs["issues"]))
    if not dfs["issues_comments"].empty:
        rows.append(_timeline_row("issues_comments", dfs["issues_comments"]))
    if not dfs["issues_events"].empty:
        rows.append(_timeline_row("issues_events", dfs["issues_events"]))
    if not dfs["prs"].empty:
        rows.append(_timeline_row("pull_requests", dfs["prs"]))
    if not dfs["prs_comments"].empty:
        rows.append(_timeline_row("pull_requests_comments", dfs["prs_comments"]))

    # (commits are already encoded in coding_history_table, so we skip them here)

    actions = pandas.DataFrame(rows, columns=["action"] + day_cols).set_index("action")

    return actions

def _load_activity_csv(folder: str,
                       filename: str,
                       rename_map: Dict[str, str],
                       dev_login,
                       usecols: list[str] = None,
                       ) -> pandas.DataFrame:
    """
    Read *filename* in *folder*, rename to the canonical columns
    ('id','date','creator_login'), keep ONLY the specified dev, and
    return three columns.  On any problem → empty df.
    """
    path = os.path.join(folder, filename)
    try:
        df = pandas.read_csv(path, sep=cfg.CSV_separator, usecols=usecols)
    except FileNotFoundError:
        logging.info("File %s not found – skipping", path)
        return pandas.DataFrame(columns=["id", "date", "creator_login"])
    except Exception as e:
        logging.warning("Could not read %s: %s", path, e)
        return pandas.DataFrame(columns=["id", "date", "creator_login"])

    df = df.rename(columns=rename_map)
    # keep only the columns we need, ignore anything extra
    df = df[["id", "date", "creator_login"]]
    df = df[df.creator_login == dev_login]
    # allow str OR list[str]
    if isinstance(dev_login, list):
        df = df[df.creator_login.isin(dev_login)]
    else:
        df = df[df.creator_login == dev_login]
    return df.reset_index(drop=True)

def get_ACTIVITY(folder: str, dev: str) -> pandas.DataFrame:

    path = os.path.join(folder, "commit_list.csv")

    # ─── load & clean ──────────────────────────────────────────────────
    df = pandas.read_csv(path, sep=cfg.CSV_separator, parse_dates=["created_at"])

    df = df[df["author_id"] == dev]           # keep only this dev
    if df.empty:                                    # no commits at all
        raise ValueError(f"No commits found for {dev}")

    dates = df["created_at"].dt.normalize()
    if getattr(dates.dt, "tz", None) is not None:
            dates = dates.dt.tz_localize(None)
    # ─── per-day aggregation ──────────────────────────────────────────
    daily_counts = (
        df.groupby(df["created_at"].dt.normalize())       # strip and remove hh:mm:ss 
          .size()
          .rename("commits")
    )

    daily_counts = (
        dates.value_counts()
             .sort_index()
             .rename("commits")
             .astype("int64")
    )
    daily_counts.index.name = "date"
    return daily_counts.to_frame()

def get_timeline(folder: str, dev: str) -> pandas.DataFrame:
    """ActivitiesExtractor.py
    This function will take all the activity and make a daily aggregated df
    """
    # actions: rows = action names, cols = day strings "YYYY-MM-DD"
    actions = get_NONCODING(folder, dev)
    actions = actions.transpose()  # make the index a DatetimeIndex


    # make the index a DatetimeIndex (tz-naive, normalized)
    actions.index = pandas.to_datetime(actions.index, errors="coerce").tz_localize(None)
    actions.index.name = "date"

    # commits: DataFrame with index=date (DatetimeIndex), col 'commits'
    commits = get_ACTIVITY(folder, dev)

    # union the date ranges and align
    # build a FULL daily index from min→max, then align
    if not actions.empty and not commits.empty:
        start = min(actions.index.min(), commits.index.min())
        end   = max(actions.index.max(), commits.index.max())
    elif not actions.empty:
        start, end = actions.index.min(), actions.index.max()
    elif not commits.empty:
        start, end = commits.index.min(), commits.index.max()
    else:
        # no activity at all → return an empty, well-typed frame
        cols = ["commits","pull_requests","issues","issues_comments",
                "issues_events","pull_requests_comments","coding_day","nc_day"]
        return pandas.DataFrame(columns=cols).astype({
            "commits":"int64","pull_requests":"int64","issues":"int64",
            "issues_comments":"int64","issues_events":"int64",
            "pull_requests_comments":"int64","coding_day":"bool","nc_day":"bool"
        })

    full_idx = pandas.date_range(start=start, end=end, freq="D")
    actions = actions.reindex(full_idx, fill_value=0)
    commits = commits.reindex(full_idx, fill_value=0)


    # merge
    df = actions.join(commits, how="outer")
    df = df.fillna(0)

    # ensure expected columns exist even if that action never occurred
    for col in ["pull_requests", "issues", "issues_comments", "issues_events", "pull_requests_comments"]:
        if col not in df.columns:
            df[col] = 0

    # booleans
    df["coding_day"] = (df["commits"] > 0)


    df["nc_day"] = (df[["issues", "issues_comments", "issues_events", "pull_requests_comments"]].sum(axis=1) > 0)

    # nice ordering
    cols = ["commits", "pull_requests", "issues", "issues_comments", "issues_events", "pull_requests_comments",
            "coding_day", "nc_day"]
    # keep any extra columns too (if you later add more)
    df = df[[c for c in cols if c in df.columns] + [c for c in df.columns if c not in cols]]
    return df.sort_index()


In [23]:
def write_pauses_table(
        df: pandas.DataFrame,
        out_path: os.PathLike,
        authors: list[str] | None = None,
        *,
        user_col: str = "author_id",
        date_col: str = "created_at",
        tail_to_today: bool = False
    ) -> pandas.DataFrame:

    df[date_col] = pandas.to_datetime(df[date_col]).dt.normalize()

    if authors is None:
        authors = df[user_col].unique()

    rows = []
    
    count =0
    for dev in authors:
        user_df = df[df[user_col] == dev]
        pause_len_1 = len(user_df[date_col].dt.date.unique())
        pause_len_2 =len(user_df)
        if user_df.empty:
            continue

        active_days = sorted(user_df[date_col].dt.date.unique())
        current_row = [dev]

        for i in range(len(active_days) - 1):
            prev_day = active_days[i]
            next_day = active_days[i + 1]
            gap = (next_day - prev_day).days
            if gap > 1:
                # Inactivity starts the day after prev_day
                current_row.append(f"{(prev_day + pandas.Timedelta(days=1)).strftime('%Y-%m-%d')}/{next_day.strftime('%Y-%m-%d')}")
            else:
                count += 1


        if tail_to_today and active_days:
            today = _date.today()
            gap = (today - active_days[-1]).days
            if gap > 1:
                current_row.append(f"{active_days[-1]}/{today}")

        if len(current_row) > 1:
            rows.append(current_row)
    
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", newline="",encoding="utf-8" ) as f:
        csv.writer(f, delimiter=",", quoting=csv.QUOTE_NONE).writerows(rows)

    return pandas.DataFrame(rows)

def get_commit_based_core_devs(commits, threshold=0.8):
    """
    commits: List[dict] where each dict contains at least the 'author' key.
    Example: [{'author': 'alice'}, {'author': 'bob'}, {'author': 'alice'}, ...]

    Returns: List of core developers (author names) who together authored >= threshold of commits.
    """
    # Count commits per developer
    author_commit_counts = Counter(commit["author_id"] for commit in commits)

    # Sort developers by number of commits (descending)
    sorted_authors = author_commit_counts.most_common()

    total_commits = sum(author_commit_counts.values())
    cumulative = 0
    core_devs = []

    for author, count in sorted_authors:
        cumulative += count
        core_devs.append(author)
        if cumulative / total_commits >= threshold:
            break

    return core_devs

def identifyInactivityPeriods(organizationFolder, organization, project):
    """Identifies the inactivity periods of the developers in the organization"""
    #url = "https://github.com/" + organization + "/" + project + ".git"
    #authors, emails = findCoreDevelopers(url, name=project)
    
    organizationFolder = organizationFolder + "/" + organization + "/" + project

    commits =  pandas.read_csv(organizationFolder + "/commit_list.csv", parse_dates=["created_at"], encoding="utf-8", header=0, sep=cfg.CSV_separator)

    commit_authors = get_commit_based_core_devs(commits.to_dict(orient='records'))

    pauses = write_pauses_table(commits, organizationFolder + "/pauses_commits.csv", commit_authors, user_col = "author_id", date_col="created_at")
    return commit_authors, pauses

def getFarOutThreshold(values, dev): ### If it is satisfying, move the function into UTILITIES
    import numpy
    th = 0
    q_3rd = numpy.percentile(values,75)
    q_1st = numpy.percentile(values,25)
    iqr = q_3rd-q_1st
    if iqr > 1:
        th = q_3rd + 3*iqr
    return th

def addToBreaksList(pauses, currentBreaks, th):
    for _, p in pauses.iterrows():
        if (p['len'] > th) and (p['dates'] not in currentBreaks.dates.tolist()):
            util.add(currentBreaks, [p['len'], p['dates'], th])
    return currentBreaks

def cleanClearBreaks(clearBreaks, breaks):
    for _, b in breaks.iterrows():
        clearBreaks = clearBreaks[clearBreaks.dates != b['dates']] # If it was in the long_breaks list, remove ot from there
    return clearBreaks

def identifyBreaks(pauses_dates_list, dev, window, shift,
                   debug_folder=None):           # NEW ARG
    '''
    Removes SURE BREAKS from windows to calculate Tfov
    and — with debug_folder — writes a per-window diagnostics CSV.
    '''
    breaks_df = pandas.DataFrame(columns=['len', 'dates', 'th'])
    diagnostics = []                             # NEW
    count = 0
    for row in pauses_dates_list:
        if row[0] != dev:              # ⬅️  ignore other developers
            continue
        
        count += 1
        if count % 50 == 0:  # Print progress every 100 developers
            print(count)
        intervals_list = [ x for x in row[1:]
                          if isinstance(x, str) and '/' in x and x.strip()]
        
        intervals_list.sort(key=lambda s: s.split('/')[0])

        if not all(a.split('/')[0] <= b.split('/')[0]
                for a, b in zip(intervals_list, intervals_list[1:])):
            print("⚠️  intervals_list UNSORTED for", dev)

        if not intervals_list:
            print(dev, 'has NO valid pauses')
            continue                      # <- don’t bail out; just skip

        clear_breaks = pandas.DataFrame(columns=['len', 'dates'])

        FPS_dt = datetime.strptime(intervals_list[0].split('/')[0], '%Y-%m-%d')
        LPE_dt = datetime.strptime(intervals_list[-1].split('/')[1], '%Y-%m-%d')

        win_start, win_end = FPS_dt, FPS_dt + timedelta(days=window)
        last_th = 0
        while win_end < LPE_dt:
            win_pauses_list = pandas.DataFrame(columns=['len', 'dates'])
            partially_included_pauses_list = pandas.DataFrame(columns=['len', 'dates'])

            for interval in intervals_list:
                int_start_str, int_end_str = interval.split('/')          # keep strings
                int_start_dt  = datetime.strptime(int_start_str, '%Y-%m-%d')
                int_end_dt    = datetime.strptime(int_end_str,   '%Y-%m-%d')
                pause_len = util.daysBetween(int_start_str, int_end_str)
                # fully inside
                if int_start_dt >= win_start and int_end_dt <= win_end:
                    util.add(win_pauses_list, [pause_len, interval])
                # touches boundary
                if ((int_start_dt <= win_end and int_end_dt > win_end) or
                    (int_end_dt >= win_start and int_start_dt < win_start)):
                    util.add(partially_included_pauses_list, [pause_len, interval])

            win_pauses = len(win_pauses_list)
            pauses = pandas.concat([win_pauses_list,
                                    partially_included_pauses_list],
                                    ignore_index=True)

            # --- decision logic (unchanged) ---------------------------------
            win_th = None
            added_flag = False
            if win_pauses >= 4:
                win_th = getFarOutThreshold(win_pauses_list['len'], dev)
                if win_th > 0:
                    before = len(breaks_df)
                    breaks_df = addToBreaksList(pauses, breaks_df, win_th)
                    added_flag = len(breaks_df) > before
                    last_th = win_th
                elif last_th > 0:
                    before = len(breaks_df)
                    breaks_df = addToBreaksList(pauses, breaks_df, last_th)
                    added_flag = len(breaks_df) > before
            else:
                if last_th > 0:
                    before = len(breaks_df)
                    breaks_df = addToBreaksList(pauses, breaks_df, last_th)
                    added_flag = len(breaks_df) > before

                clear_breaks = cleanClearBreaks(clear_breaks, breaks_df)
                for _, p in pauses.iterrows():
                    if (p['len'] >= window and
                        p['dates'] not in clear_breaks.dates.tolist() and
                        p['dates'] not in breaks_df.dates.tolist()):
                        util.add(clear_breaks, p)

            # ----------- NEW: record diagnostics for this window -------------
            diagnostics.append({
                'win_start': win_start.date(),
                'win_end':   win_end.date(),
                'win_pauses': win_pauses,
                'pause_lengths': ';'.join(map(str, win_pauses_list['len'].tolist())),
                'partial_lengths': ';'.join(map(str, partially_included_pauses_list['len'].tolist())),
                'win_th': win_th,
                'last_th': last_th,
                'added_as_break': 'yes' if added_flag else 'no'
            })
            # -----------------------------------------------------------------

            win_start += timedelta(days=shift)
            win_end   = win_start + timedelta(days=window)


    return breaks_df

### LABEL BREAKS

In [20]:
def label_developers_activity() -> pandas.DataFrame:
    """
    main function for labeling developers

    sets up varables to call label_timeline

    """
    #identifyInactivityPeriods
    #"Resources/repositories.txt"
    repos_file= '../' + cfg.repos_file
    #"../Organizations"
    organizationFolder = cfg.main_folder


    #identifyBreaks
    win = cfg.sliding_window_size
    shift = cfg.shift

    tf_devs= []

    with open(repos_file) as f:
        repos_line = f.readlines()
        for repo in repos_line:
            #take the end '\n' out
            repo = repo.rstrip('\n')
            organization, project = repo.split('/')

            print(f"Start Identifying inactivity periods for {organization}/{project}...")

            authors, pauses = identifyInactivityPeriods( organizationFolder, organization, project)

            #make pauses to a csv file at this location C:\Users\samut\OneDrive\Documents\GitHub\developersInactivityAnalysisCOPY\Organizations\Rdatatable\data.table\Results
        
            pauses_list = pauses.values.tolist()
            print(f"{len(authors)} Developers inactivity periods identified")

            output_folder = organizationFolder + '/' + repo + "/Results"
            os.makedirs(output_folder, exist_ok=True)
            
            for dev in authors:
                tf_devs.append(dev)
                timeline_folder = organizationFolder + '/' + repo + '/' + cfg.timeline_folder_name
                os.makedirs(timeline_folder, exist_ok=True)
            
                timeline_path = Path(timeline_folder) / f"{dev}_timeline.csv"

                if timeline_path.is_file():
                    user_timeline = pandas.read_csv(timeline_path, sep=cfg.CSV_separator, index_col=0)
                else:
                    folder = organizationFolder + '/' + repo
                    user_timeline = get_timeline(folder, dev)

                    #transpose the user_timeline making the frist row the first column
                    user_timeline.to_csv(timeline_path, sep=cfg.CSV_separator, na_rep=cfg.CSV_missing, quoting=None, lineterminator='\n')

                print(dev)
            
                #make a break folder 
                breaks_folder = organizationFolder + '/' + repo + "/Breaks"
                os.makedirs(breaks_folder, exist_ok=True)
        
                breaks_path =  Path(breaks_folder)/  f"{dev}_breaks.csv"              

                if breaks_path.is_file():
                    breaks_df = pandas.read_csv(breaks_path, sep=cfg.CSV_separator, index_col=0)  

                else:
                    breaks_df = pandas.DataFrame(columns=['len', 'dates', 'th'])
                    breaks_df = identifyBreaks(pauses_list, dev=dev, window=win, shift=shift, debug_folder=output_folder )
                    breaks_df.to_csv(breaks_path, sep=cfg.CSV_separator, na_rep=cfg.CSV_missing, index=False, lineterminator="\n")
                            
                #add label_timeline
                user_timeline = label_timeline(user_timeline, breaks_df)

                out_csv = Path(output_folder) / f"{dev}_labeled_timeline.csv"
                user_timeline.to_csv(out_csv,
                                      sep=cfg.CSV_separator, na_rep=cfg.CSV_missing, quoting=None, lineterminator='\n', index_label='date')

                tf_devs_df = pandas.DataFrame(tf_devs, columns=["developer"])
                tf_devs_df.to_csv(Path(output_folder) / "tf_devs.csv", sep=cfg.CSV_separator, na_rep=cfg.CSV_missing, quoting=None, lineterminator='\n')

    return tf_devs

def label_timeline(user_timeline, breaks_df):
    """
    Make a labled timeline of devlopers breaks

    given a user_timeline and breaks_df
    user_timeline
    commits,pull_requests,issues,issues_comments,issues_events,pull_requests_comments,coding_day,nc_day
    0,0,2,1,0,0,False,True
    3,0,0,0,0,0,True,False
    0,0,0,0,0,0,False,False

    breaks_df
    len,dates,th
    72,2015-05-05/2015-07-16,59.25
    """
    df = user_timeline.copy()

    df["break_day"] = None

    #this marks the break days from breaks_df onto user_timeline
    for breaks in breaks_df.itertuples():
        start = breaks.dates.split('/')[0]
        end = breaks.dates.split('/')[1]

        if start in df.index and bool(df.at[start, "coding_day"]):
            start = start + pandas.Timedelta(days=1)

        end = pandas.to_datetime(end) - pandas.Timedelta(days=1)

        break_range = pandas.date_range(start=pandas.to_datetime(start)+pandas.Timedelta(days=1),
                                end=pandas.to_datetime(end)-pandas.Timedelta(days=1))

        if pandas.to_datetime(start) <= end:

           for date in break_range:
                date = date.strftime("%Y-%m-%d")
                df.at[date, "break_day"] = True
                df.at[date, "th"] = breaks.th
                df.at[date, "len"] = breaks.Index

    #turns all the NA into false
    for col in ["coding_day", "nc_day", "break_day"]:
        df[col] = df[col].fillna(False).astype("bool")


    df.index = pandas.to_datetime(df.index)
    df = df.sort_index()

    # Optional: unmark the break *end* day (commit day) so it’s not counted as break
    # break end - 1 day = end non coding
    for breaks in breaks_df.itertuples():
        end = pandas.to_datetime(breaks.dates.split('/')[1])
        if end in df.index:
            df.at[end, "break_day"] = False

    gone_days = 365

    df["event_day"] = df["coding_day"] | df["nc_day"]

    df["state"] = "ACTIVE"

    # Identify contiguous break windows (groups of consecutive True in break_day)
    bd = df["break_day"]
    group_id = (bd != bd.shift(1)).cumsum()

    # Precompute last event BEFORE a given date (global, across timeline)
    all_events_idx = df.index[df["event_day"]]

    for gid, block in df.groupby(group_id):
        if not block["break_day"].iloc[0]:
            continue  # not a break chunk

        # This is one contiguous break [start .. end] (inclusive)
        start_ts = block.index[0]
        end_ts   = block.index[-1]

        # Lookahead info: is there any non-coding event in this break?
        has_nc = bool((df.loc[start_ts:end_ts, "nc_day"]).any())

        th_vals = df.loc[start_ts:end_ts, "th"].dropna()
        Tfov = int(round(th_vals.iloc[0])) if not th_vals.empty else 14

        # Anchor silence to the last event (coding or non-coding) before the break starts
        prev_nc_idx = df.index[df["nc_day"] & (df.index < start_ts)]
        last_nc_before = prev_nc_idx.max() if len(prev_nc_idx) else None

        last_nc = None  # most recent non-coding event inside this break
        if last_nc_before is not None and (start_ts - last_nc_before).days <= Tfov:
            # Seed the NON_CODING hold across the start of the break
            last_nc = last_nc_before

        # Walk day by day inside the break
        for d in block.index:

            # Non-coding event day => NON_CODING and update last_nc
            if bool(df.at[d, "nc_day"]):
                df.at[d, "state"] = "NON_CODING"
                last_nc = d
                continue

            # Silent day inside a break -> decide via Tfov and gone
            # Compute silence since the most relevant last event:
            # - Prefer last NC inside break; else use last event before break; else start-of-break as approximate anchor.
            ref_nc = last_nc if last_nc is not None else None
            if (ref_nc is not None) and ((d - ref_nc).days <= Tfov):
                df.at[d, "state"] = "NON_CODING"
                continue

            # No recent NC: INACTIVE vs GONE (since last ANY event)
            last_any = ref_nc if ref_nc is not None else (all_events_idx[all_events_idx < start_ts].max() if len(all_events_idx[all_events_idx < start_ts]) else None)
            silent_days = (d - last_any).days
            if silent_days > gone_days:
                df.at[d, "state"] = "GONE"
            else:
                df.at[d, "state"] = "INACTIVE"


    return df

tf_devs = label_developers_activity()


Start Identifying inactivity periods for Rdatatable/data.table...
6 Developers inactivity periods identified
mattdowle
MichaelChirico
jangorecki
arunsrinivasan
ben-schwen
tdhock


Treats all non-break rows as ACTIVE

# Prediction

In [35]:
def users_activity_concat(authors):
    repos_file= '../' + cfg.repos_file
    
    frames = []

    with open(repos_file) as f:
        repos_line = f.readlines()
        for repo in repos_line:
            repo = repo.rstrip('\n')
            base = cfg.main_folder + "/" + repo + "/" + cfg.results_folder
            for dev in authors:
                file_name = Path(base) / f"{dev}_labeled_timeline.csv"
                if not file_name.exists():
                    # Optional: log this instead of printing
                    continue
                #parse date is the index column
                df = pandas.read_csv(
                    file_name, parse_dates=["date"]
                )

                df["author"] = dev
                df["repo"] = repo

                df["date"] = df["date"].dt.normalize()

                df = df.sort_values("date")

                frames.append(df)

    out = pandas.concat(frames, ignore_index=True, sort=False)
    out = out.sort_values(["author", "repo", "date"]).reset_index(drop=True)

    return out

In [36]:
def segmentize_timeline(
    daily: pandas.DataFrame,
    state_order: List[str] = ("ACTIVE", "NON_CODING", "INACTIVE", "GONE"),
) -> Tuple[pandas.DataFrame, pandas.DataFrame]:
    """
    Convert a per-day timeline into consecutive same-state segments per (author, repo).

    Inputs
    ------
    daily : DataFrame with at least columns:
        ['author','repo','date','state',
         'commits','pull_requests','issues','issues_comments',
         'issues_events','pull_requests_comments','coding_day','nc_day','break_day','th','len','event_day']
        (extras are okay)

    state_order : list of states in desired categorical order.

    Returns
    -------
    daily_with_segments : original daily rows +:
        'segment_id'  (int, 0-based within each (author,repo))
        'segment_pos' (1-based position within the segment)
        'segment_len' (length of the segment in days)

    segments : one row per segment with keys & aggregates:
        ['author','repo','segment_id','state_curr','start_date','end_date','seg_len',
         'commits_sum','pr_sum','issues_sum','issues_comments_sum','issues_events_sum','pr_comments_sum',
         'coding_days','nc_days','break_days','prev_state','prev_seg_len','next_state']
    """
    df = daily.copy()

    # --- Normalize key types & sort deterministically ---
    df["author"] = df["author"].astype(str).str.strip()
    df["repo"]   = df["repo"].astype(str).str.strip()
    df["date"]   = pandas.to_datetime(df["date"], utc=True, errors="coerce")
    df["state"]  = pandas.Categorical(df["state"].astype(str),
                                  categories=list(state_order), ordered=True)

    df = df.sort_values(["author","repo","date"], kind="stable").reset_index(drop=True)

    # --- Compute per-(author,repo) segment boundaries ---
    # A new segment starts whenever state != previous state's value (or first row)
    grp = df.groupby(["author","repo"], sort=False, observed=True)
    first_in_group = grp.cumcount().eq(0)
    state_changed  = df["state"].ne(grp["state"].shift(1))
    prev_state = grp["state"].shift(1)
    next_state = grp["state"].shift(-1)
    one_day_flip = prev_state.eq(next_state) & df["state"].ne(prev_state)
    df.loc[one_day_flip, "state"] = prev_state[one_day_flip]
    is_seg_start   = first_in_group | state_changed

    # segment_id: 0,1,2,... within each (author,repo)
    df["segment_id"] = grp.apply(lambda g: is_seg_start.loc[g.index].cumsum() - 1).reset_index(level=[0,1], drop=True)

    # Position & length inside segment (handy during analysis or later expansions)
    df["segment_pos"] = df.groupby(["author","repo","segment_id"], observed=True).cumcount() + 1
    df["segment_len"] = df.groupby(["author","repo","segment_id"], observed=True)["date"].transform("size")

    # --- Build segment-level table with aggregates ---
    agg_map = {
        "date": ["min","max","size"],
        "commits": "sum",
        "pull_requests": "sum",
        "issues": "sum",
        "issues_comments": "sum",
        "issues_events": "sum",
        "pull_requests_comments": "sum",
        "coding_day": "sum",
        "nc_day": "sum",
        "break_day": "sum",
    }

    seg = (df
           .groupby(["author","repo","segment_id","state"], observed=True, as_index=False)
           .agg(start_date=("date","min"),
                end_date=("date","max"),
                seg_len=("date","size"),
                commits_sum=("commits","sum"),
                pr_sum=("pull_requests","sum"),
                issues_sum=("issues","sum"),
                issues_comments_sum=("issues_comments","sum"),
                issues_events_sum=("issues_events","sum"),
                pr_comments_sum=("pull_requests_comments","sum"),
                coding_days=("coding_day","sum"),
                nc_days=("nc_day","sum"),
                break_days=("break_day","sum"))
          )
    seg.rename(columns={"state": "state_curr"}, inplace=True)

    # Previous/next segment context (per author,repo)
    seg = seg.sort_values(["author","repo","segment_id"], kind="stable")
    seg["prev_state"]   = seg.groupby(["author","repo"], observed=True)["state_curr"].shift(1)
    seg["prev_seg_len"] = seg.groupby(["author","repo"], observed=True)["seg_len"].shift(1)
    seg["next_state"]   = seg.groupby(["author","repo"], observed=True)["state_curr"].shift(-1)

    # Keep date columns as date (drop TZ & normalize to date if you prefer)
    # seg["start_date"] = seg["start_date"].dt.tz_localize(None).dt.date
    # seg["end_date"]   = seg["end_date"].dt.tz_localize(None).dt.date

    return df, seg


In [37]:
def add_predictors(seg: pandas.DataFrame) -> pandas.DataFrame:
    """
    Segment-level feature builder for next-segment-state prediction.

    Input  : daily timeline with columns >=
             ['author','repo','date','state','commits','pull_requests','issues',
              'issues_comments','issues_events','pull_requests_comments',
              'coding_day','nc_day','break_day','th','len','event_day']

    Output : segment table with features and targets:
             - keys:   author, repo, segment_id, start_date, end_date
             - labels: state_t, target_next_state
             - features: seg_len, densities, proportions, prev_* and rolling stats
             - a 'date' column (== start_date) kept for downstream compatibility
    """
    # 1) segmentize

    if seg.empty:
        return pandas.DataFrame(columns=[
            "author","repo","segment_id","date","start_date","end_date",
            "state_t","target_next_state"
        ])

    # 2) core, leakage-safe features (predicting at segment END)
    out = seg.copy()

    # densities / proportions within the segment
    eps = 1e-9
    out["avg_commits_per_day"]   = out["commits_sum"] / (out["seg_len"] + eps)
    out["avg_prs_per_day"]       = out["pr_sum"] / (out["seg_len"] + eps)
    out["avg_issues_per_day"]    = out["issues_sum"] / (out["seg_len"] + eps)
    out["avg_comments_per_day"]  = out["issues_comments_sum"] / (out["seg_len"] + eps)
    out["avg_events_per_day"]    = out["issues_events_sum"] / (out["seg_len"] + eps)
    out["avg_pr_comments_per_day"] = out["pr_comments_sum"] / (out["seg_len"] + eps)

    out["pct_coding_days"]    = out["coding_days"] / (out["seg_len"] + eps)
    out["pct_noncoding_days"] = out["nc_days"]     / (out["seg_len"] + eps)
    out["pct_break_days"]     = out["break_days"]  / (out["seg_len"] + eps)

    # calendar at segment start
    out["start_date"] = pandas.to_datetime(out["start_date"], utc=True, errors="coerce")
    out["start_dow"]  = out["start_date"].dt.dayofweek  # 0=Mon
    out["start_woy"]  = out["start_date"].dt.isocalendar().week.astype(int)
    out["start_month"]= out["start_date"].dt.month
    out["start_qtr"]  = ((out["start_month"] - 1) // 3 + 1).astype(int)

    # prior segment context (already present): prev_state, prev_seg_len
    # simple bigram feature
    out["prev_curr_pair"] = (out["prev_state"].astype(str) + "→" + out["state_curr"].astype(str))

    # author/repo-level rolling stats on previous segments (k = 3, 5)
    by = ["author","repo"]
    for k in (3, 5):
        out[f"roll{k}_seg_len_mean"]   = _roll_feat(out, by, "seg_len", k, "mean")
        out[f"roll{k}_seg_len_std"]    = _roll_feat(out, by, "seg_len", k, "std")
        out[f"roll{k}_commits_mean"]   = _roll_feat(out, by, "commits_sum", k, "mean")
        out[f"roll{k}_nc_days_mean"]   = _roll_feat(out, by, "nc_days", k, "mean")
        out[f"roll{k}_break_days_mean"]= _roll_feat(out, by, "break_days", k, "mean")

    # 3) labels and presentation
    out.rename(columns={"state_curr": "state_t"}, inplace=True)
    out["state_t"] = out["state_t"].astype(str)
    out["target_next_state"] = out["next_state"].astype(str)

    # keep only rows with a known next segment (drop last segment per series)
    out = out[out["target_next_state"].notna()].copy()

    # compatibility: expose a 'date' column (use segment start date)
    out["date"] = out["start_date"]

    # keys to keep for later joins/inspection
    keep_first = ["author","repo","segment_id","date","start_date","end_date","state_t","target_next_state"]
    # rest are features
    feat_cols = [c for c in out.columns if c not in keep_first + ["next_state"]]

    # reorder nicely: keys/labels first, then features
    out = out[keep_first + feat_cols].sort_values(["author","repo","segment_id"]).reset_index(drop=True)
    return out

def _roll_feat(seg: pandas.DataFrame, by_cols: List[str], col: str, k: int, fn: str) -> pandas.Series:
    """Groupwise rolling (on previous segments only)."""
    s = (seg
         .groupby(by_cols, observed=True)[col]
         .apply(lambda x: getattr(x.shift(1).rolling(k, min_periods=1), fn)()))
    # the groupby/apply preserves a hierarchical index; align back:
    return s.reset_index(level=by_cols, drop=True)

def tail_features(daily_with_segments, ks=(3,7,14)):
    d = daily_with_segments.sort_values(["author","repo","segment_id","date"]).copy()
    d["any_event"] = (
        d[["commits","pull_requests","issues","issues_comments","issues_events","pull_requests_comments"]]
        .sum(axis=1) > 0
    ).astype(int)

    out = d[["author","repo","segment_id"]].drop_duplicates().copy()
    g = d.groupby(["author","repo","segment_id"], observed=True)

    for k in ks:
        tail_ev = g["any_event"].apply(lambda s: s.tail(k).mean()).reset_index(level=[0,1], drop=True)
        tail_cd = g["coding_day"].apply(lambda s: s.tail(k).mean()).reset_index(level=[0,1], drop=True)
        out[f"tail{k}_event_rate"] = tail_ev.values
        out[f"tail{k}_coding_rate"] = tail_cd.values
    return out

In [38]:
STATE_ORDER = ["ACTIVE", "NON_CODING", "INACTIVE", "GONE"]  #


@dataclass
class SplitConfig:
    strategy: str = "holdout_authors"   # "time_by_repo" | "time_global" | "holdout_authors"
    val_months: int = 6              # last N months per repo for validation
    holdout_fraction: float = 0.2    # for holdout_authors

def make_splits(df: pandas.DataFrame, cfg: SplitConfig) -> tuple[np.ndarray, np.ndarray]:
    """
    Returns boolean masks (train_mask, val_mask) aligned to df rows.
    """
    idx = np.arange(len(df))
    train_mask = np.zeros(len(df), dtype=bool)
    val_mask   = np.zeros(len(df), dtype=bool)

    if cfg.strategy == "time_by_repo":
        for (a, r), g in df.groupby(["author","repo"], sort=False):
            if g.empty: continue
            last_date = g["date"].max()
            cutoff = last_date - pandas.Timedelta(days=cfg.val_months*30)
            sel_train = g["date"] <= cutoff
            sel_val   = g["date"] >  cutoff
            train_mask[g.index] = sel_train.values
            val_mask[g.index]   = sel_val.values

    elif cfg.strategy == "time_global":
        cutoff = df["date"].max() - pandas.Timedelta(days=cfg.val_months*30)
        train_mask = (df["date"] <= cutoff).values
        val_mask   = ~train_mask

    elif cfg.strategy == "holdout_authors":
        val_mask   = df["author"].isin(["jangorecki"]).values
        train_mask = ~val_mask
    else:
        raise ValueError(f"Unknown split strategy: {cfg.strategy}")

    # Ensure both non-empty
    if not train_mask.any() or not val_mask.any():
        raise RuntimeError("Empty train or val split; adjust SplitConfig.")

    return train_mask, val_mask

def _normalize_state_series(s: pandas.Series) -> pandas.Series:
    # make robust to stray variants like "Non-Coding", "gone ", etc.
    norm = (s.astype(str)
              .str.strip()
              .str.upper()
              .str.replace("-", "_"))
    # map anything unexpected to NaN so we drop it later
    norm = norm.where(norm.isin(STATE_ORDER), other=np.nan)
    return norm

def add_predictors_segments(segments: pandas.DataFrame) -> pandas.DataFrame:
    """
    Build segment-level features for next-segment-state prediction.
    Input: segments from segmentize_timeline()
    Output: 1 row per segment (excluding final segments without a next_state).
    """
    seg = segments.copy()

    # normalize states
    seg["state_curr"] = _normalize_state_series(seg["state_curr"])
    if "next_state" in seg.columns:
        seg["next_state"] = _normalize_state_series(seg["next_state"])

    # drop rows without a defined next segment (last segment in a series)
    if "next_state" in seg.columns:
        seg = seg[seg["next_state"].notna()].copy()
    else:
        raise ValueError("segments must include 'next_state' to define the target.")

    # densities/proportions
    eps = 1e-9
    seg["avg_commits_per_day"]     = seg["commits_sum"] / (seg["seg_len"] + eps)
    seg["avg_prs_per_day"]         = seg["pr_sum"] / (seg["seg_len"] + eps)
    seg["avg_issues_per_day"]      = seg["issues_sum"] / (seg["seg_len"] + eps)
    seg["avg_comments_per_day"]    = seg["issues_comments_sum"] / (seg["seg_len"] + eps)
    seg["avg_events_per_day"]      = seg["issues_events_sum"] / (seg["seg_len"] + eps)
    seg["avg_pr_comments_per_day"] = seg["pr_comments_sum"] / (seg["seg_len"] + eps)

    seg["pct_coding_days"]    = seg["coding_days"] / (seg["seg_len"] + eps)
    seg["pct_noncoding_days"] = seg["nc_days"]     / (seg["seg_len"] + eps)
    seg["pct_break_days"]     = seg["break_days"]  / (seg["seg_len"] + eps)

    # calendar at segment start
    seg["start_date"] = pandas.to_datetime(seg["start_date"], utc=True, errors="coerce")
    seg["start_dow"]   = seg["start_date"].dt.dayofweek
    seg["start_month"] = seg["start_date"].dt.month
    seg["start_qtr"]   = ((seg["start_month"] - 1) // 3 + 1).astype(int)

    # previous context
    if "prev_state" not in seg.columns:
        seg["prev_state"] = np.nan
    if "prev_seg_len" not in seg.columns:
        seg["prev_seg_len"] = np.nan

    seg["prev_curr_pair"] = (seg["prev_state"].astype(str)
                             + "→" + seg["state_curr"].astype(str))

    # labels + compatibility
    seg.rename(columns={"state_curr": "state_t"}, inplace=True)
    seg["target_next_state"] = seg["next_state"].astype(str)
    seg["date"] = seg["start_date"]  # keep a 'date' column for split code

    # ensure keys exist
    needed = ["author","repo","segment_id","state_t","target_next_state","date"]
    missing = [c for c in needed if c not in seg.columns]
    if missing:
        raise ValueError(f"segments missing required columns: {missing}")

    # sort and return
    seg = seg.sort_values(["author","repo","segment_id"]).reset_index(drop=True)
    return seg

def prepare_ml_tables_segments(df: pandas.DataFrame) -> Tuple[pandas.DataFrame, np.ndarray, List[str]]:
    """
    Build X (features) and y (labels) for NEXT-SEGMENT-STATE classification.
    Returns (X_df, y_np, feature_cols).
    """
    df = df.copy()

    # categorical features to one-hot
    cat_cols = ["state_t", "prev_state", "prev_curr_pair"]  # optional: "author","repo"
    for c in cat_cols:
        if c not in df.columns:
            df[c] = "∅"

    X_cat = pandas.get_dummies(df[cat_cols].astype("category"), prefix=cat_cols, dummy_na=False)

    # numeric features (ensure present; fill missing with 0)
    num_cols = [
        "seg_len",
        "avg_commits_per_day","avg_prs_per_day","avg_issues_per_day",
        "avg_comments_per_day","avg_events_per_day","avg_pr_comments_per_day",
        "pct_coding_days","pct_noncoding_days","pct_break_days",
        "start_dow","start_month","start_qtr",
        "prev_seg_len", "tail"
        # rolling segment stats if you added them (safe if absent):
        "roll3_seg_len_mean","roll3_seg_len_std",
        "roll3_commits_mean","roll3_nc_days_mean","roll3_break_days_mean",
        "roll5_seg_len_mean","roll5_seg_len_std",
        "roll5_commits_mean","roll5_nc_days_mean","roll5_break_days_mean",
    ]
    for c in num_cols:
        if c not in df.columns:
            df[c] = 0.0
    X_num = df[num_cols].astype(float)

    # concat features
    X = pandas.concat([X_num, X_cat], axis=1)
    feature_cols = list(X.columns)

    # labels — map to 0..K-1 and DROP any -1 rows
    cat = pandas.Categorical(df["target_next_state"], categories=STATE_ORDER, ordered=True)
    y = cat.codes.astype(np.int64)

    valid = y >= 0
    if not valid.all():
        X = X.loc[valid].reset_index(drop=True)
        y = y[valid]
    return X, y, feature_cols, valid

def train_classifier(X_tr, y_tr, X_val, y_val):
    K = len(STATE_ORDER)
    y_tr = np.asarray(y_tr, dtype=np.int64)
    y_val = np.asarray(y_val, dtype=np.int64)

    class_counts = np.bincount(y_tr, minlength=K)
    total = class_counts.sum()
    weights = {i: (total / (K * max(1, int(c)))) for i, c in enumerate(class_counts)}
    sample_w = np.vectorize(weights.get)(y_tr)

    model = XGBClassifier(
        n_estimators=2000,              # allow ES to pick best
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=2.0,
        objective="multi:softprob",
        num_class=K,
        random_state=42,
        tree_method="hist",
        n_jobs=-1,                      # use all cores
        eval_metric=["mlogloss","merror"]
    )
    model.fit(
        X_tr, y_tr,
        sample_weight=sample_w,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    return model

def attach_predictions_to_segments(
    segments_raw: pandas.DataFrame,      # output of segmentize_timeline(...)[1]
    features_df: pandas.DataFrame,       # output of add_predictors(segments or daily)
    X,                                # features matrix aligned to features_df
    model,                            # trained classifier
    val_mask,                         # boolean mask over features_df for held-out rows
    state_order: List[str],           # ["ACTIVE","NON_CODING","INACTIVE","GONE"]
    out_csv_path: str,
    *,
    filter_to_val_authors: bool = True,
    only_author: Optional[str] = None # e.g., "jangorecki"
) -> pandas.DataFrame:
    """
    Merge next-state predictions back onto the *segment table* (not daily).
    Keeps one row per (author,repo,segment_id). Filters to held-out author(s) by default.
    """

    # --- sanity: must have segment keys ---
    required_keys = {"author","repo","segment_id"}
    if not required_keys.issubset(segments_raw.columns):
        raise ValueError(f"segments_raw missing keys: {required_keys - set(segments_raw.columns)}")
    if not required_keys.issubset(features_df.columns):
        raise ValueError(f"features_df missing keys: {required_keys - set(features_df.columns)}")

    # --- who is held out? ---
    heldout_authors = (
        features_df.loc[val_mask, "author"]
        .astype(str).str.strip().dropna().unique().tolist()
    )
    if only_author:
        heldout_authors = [only_author]

    # --- predict on held-out rows in *features_df* ---
    X_val = X[val_mask]
    yhat = model.predict(X_val)
    try:
        proba = model.predict_proba(X_val)
    except Exception:
        proba = None

    preds = features_df.loc[val_mask, ["author","repo","segment_id","state_t","target_next_state"]].copy()
    preds.rename(columns={"target_next_state": "actual_next_state"}, inplace=True)

    X_val = X[val_mask]
    yhat = model.predict(X_val)
    proba = model.predict_proba(X_val)

    # blend with prior from TRAIN rows only
    P_prior = transition_prior(features_df, train_mask=~val_mask, states=state_order, alpha=1.0)
    lam = 0.35   # small prior weight (tune 0.2–0.5)
    rows = []
    for st, p in zip(preds["state_t"].to_numpy(), proba):
        prior = P_prior.loc[st].to_numpy()
        rows.append((1-lam)*p + lam*prior)
    proba = np.vstack(rows)

    # write calibrated columns as before
    for i, name in enumerate(state_order):
        preds[f"p_{name.lower()}"] = proba[:, i]
    preds["pred_next_state"] = pandas.Categorical.from_codes(
        yhat, categories=state_order, ordered=True
    ).astype(str)
    preds["pred_confidence"] = proba.max(axis=1)

    # --- normalize join keys & filter left table to held-out authors if requested ---
    def _norm(df: pandas.DataFrame) -> pandas.DataFrame:
        out = df.copy()
        out["author"] = out["author"].astype(str).str.strip()
        out["repo"]   = out["repo"].astype(str).str.strip()
        out["segment_id"] = pandas.to_numeric(out["segment_id"], errors="coerce").astype("Int64")
        return out

    left  = _norm(segments_raw)
    right = _norm(preds)

    if filter_to_val_authors:
        left = left[left["author"].isin(heldout_authors)].copy()

    # --- merge segment-level predictions back onto the raw segments ---
    keep_cols = ["author","repo","segment_id","pred_next_state","actual_next_state"]
    if proba is not None:
        keep_cols += [f"p_{n.lower()}" for n in state_order] + ["pred_confidence"]

    right = right[keep_cols].drop_duplicates(subset=["author","repo","segment_id"])

    seg_out = left.merge(
        right, on=["author","repo","segment_id"], how="left", validate="one_to_one"
    )

    # --- correctness (NaN for last segments that naturally have no next state) ---
    seg_out["correct_next"] = (
        seg_out["pred_next_state"].notna() &
        (seg_out["pred_next_state"] == seg_out["actual_next_state"])
    )

    # --- nice column order: keys, timing, current-state, then predictions ---
    front = [
        "author","repo","segment_id",
        "state_curr","start_date","end_date","seg_len",
        "commits_sum","pr_sum","issues_sum","issues_comments_sum","issues_events_sum","pr_comments_sum",
        "coding_days","nc_days","break_days",
        # prev context if present
        *([c for c in ["prev_state","prev_seg_len"] if c in seg_out.columns]),
        # predictions
        "pred_next_state","actual_next_state","correct_next"
    ]
    prob_cols = [c for c in seg_out.columns if c.startswith("p_")] + (["pred_confidence"] if "pred_confidence" in seg_out.columns else [])
    # keep any other columns at the end
    remaining = [c for c in seg_out.columns if c not in set(front + prob_cols)]
    ordered_cols = [c for c in front if c in seg_out.columns] + prob_cols + remaining

    seg_out = seg_out[ordered_cols].sort_values(["author","repo","segment_id"]).reset_index(drop=True)

    seg_out.to_csv(out_csv_path, index=False)
    view_df(seg_out, "seg_out")                 # optional

    return seg_out

def transition_prior(df, train_mask, states=STATE_ORDER, alpha=1.0):
    # counts of state_t -> target_next_state over TRAIN rows only
    t = (df.loc[train_mask, ["state_t","target_next_state"]]
           .dropna()
           .value_counts()
           .rename("n")
           .reset_index())
    M = pandas.DataFrame(alpha, index=states, columns=states, dtype=float)
    for _, r in t.iterrows():
        M.loc[str(r["state_t"]), str(r["target_next_state"])] += r["n"]
    P = M.div(M.sum(axis=1), axis=0)   # row-normalize
    return P


In [39]:
def predict_state(authors: list[str], split_cfg: SplitConfig = SplitConfig()):
    STATE_ORDER = ["ACTIVE", "NON_CODING", "INACTIVE", "GONE"]

    # 1) LOAD
    raw = users_activity_concat(authors)

    # 1a) SEGMENTIZE
    daily_with_segments, segments = segmentize_timeline(raw, state_order=STATE_ORDER)

    tf = tail_features(daily_with_segments)
    segments = segments.merge(tf, on=["author","repo","segment_id"], how="left")

    # 2) FEATURES (segment-level)
    data = add_predictors_segments(segments)  # << updated

    # 3) SPLITS
    train_mask, val_mask = make_splits(data, split_cfg)

    # 3b) TABLES (segment-level)
    X, y, feature_cols, valid = prepare_ml_tables_segments(data)

    train_mask2 = train_mask & valid
    val_mask2   = val_mask   & valid

    X_tr, y_tr = X.loc[train_mask2], y[train_mask2]
    X_val, y_val = X.loc[val_mask2], y[val_mask2]

    # 4) TRAIN
    model = train_classifier(X_tr, y_tr, X_val, y_val)

    # 5) ATTACH TO SEGMENTS (held-out author only)
    out_csv = r"C:\Users\samut\OneDrive\Documents\GitHub\developersInactivityAnalysisCOPY\PredictionModel\predictions_segments.csv"
    out = attach_predictions_to_segments(
        segments_raw=segments,
        features_df=data,
        X=X.values,                     # XGB accepts numpy arrays; you can also pass DataFrame
        model=model,
        val_mask=val_mask,
        state_order=STATE_ORDER,
        out_csv_path=out_csv,
        filter_to_val_authors=True,
        only_author="jangorecki"
    )
    out["correct_next"].value_counts()
    #count wrong state predictions and which state was predicted wrong
    wrong_predictions = out[out["correct_next"] == False]
    wrong_counts = wrong_predictions["pred_next_state"].value_counts()
    correct_predictions = out[out["correct_next"] == True]
    correct_counts = correct_predictions["pred_next_state"].value_counts()
    print("Correct state predictions:", correct_counts)
    print("Wrong state predictions:", wrong_counts)


    y_true = pandas.Categorical(out["actual_next_state"], categories=STATE_ORDER, ordered=True).codes
    y_pred = pandas.Categorical(out["pred_next_state"],  categories=STATE_ORDER, ordered=True).codes

    print(classification_report(y_true, y_pred, target_names=STATE_ORDER, zero_division=0))
    print(pandas.DataFrame(confusion_matrix(y_true, y_pred),
                        index=[f"true_{s}" for s in STATE_ORDER],
                        columns=[f"pred_{s}" for s in STATE_ORDER]))

    # top-2 accuracy (uses your p_* columns)
    P = out[[f"p_{s.lower()}" for s in STATE_ORDER]].to_numpy()
    top2 = np.argsort(P, axis=1)[:, -2:]
    top2_hit = np.array([y_true[i] in top2[i] for i in range(len(y_true))])
    print("Top-1 acc:", (y_true==y_pred).mean(), " | Top-2 acc:", top2_hit.mean())

    #rearrange rows
    #i want the order to be segment_id, author, state_curr, pred_next_state,	actual_next_state,	correct_next
    out = out[["segment_id", "author", "state_curr", "pred_next_state", "actual_next_state", "correct_next", "repo", "start_date", "end_date", "seg_len", "commits_sum", "pr_sum", "issues_sum", "issues_comments_sum", "issues_events_sum", "pr_comments_sum", "coding_days", "nc_days", "break_days", "prev_state", "prev_seg_len", "p_active", "p_non_coding", "p_inactive", "p_gone", "pred_confidence", "next_state"]]

    view_df(out, "seg_out")
    out.to_csv(r"C:\Users\samut\OneDrive\Documents\GitHub\developersInactivityAnalysisCOPY\PredictionModel\predictions_segments.csv", index=False)


    return out

STATE_ORDER = ["ACTIVE", "NON_CODING", "INACTIVE", "GONE"]  
# read this csv"C:\Users\samut\OneDrive\Documents\GitHub\developersInactivityAnalysisCOPY\Organizations\Rdatatable\data.table\Results\tf_devs.csv"
authors = pandas.read_csv(r"C:\Users\samut\OneDrive\Documents\GitHub\developersInactivityAnalysisCOPY\Organizations\Rdatatable\data.table\Results\tf_devs.csv")
authors = authors["developer"].tolist()
print(authors)
split_cfg = SplitConfig()

out = predict_state(authors, split_cfg)

['mattdowle', 'MichaelChirico', 'jangorecki', 'arunsrinivasan', 'ben-schwen', 'tdhock']


C:\Users\samut\AppData\Local\Temp\ipykernel_34428\1705973146.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df["segment_id"] = grp.apply(lambda g: is_seg_start.loc[g.index].cumsum() - 1).reset_index(level=[0,1], drop=True)


Correct state predictions: pred_next_state
ACTIVE        28
INACTIVE      22
NON_CODING    19
Name: count, dtype: int64
Wrong state predictions: pred_next_state
INACTIVE      8
ACTIVE        6
NON_CODING    1
Name: count, dtype: int64
              precision    recall  f1-score   support

      ACTIVE       1.00      1.00      1.00         1
  NON_CODING       0.82      0.85      0.84        33
    INACTIVE       0.95      0.70      0.81        27
        GONE       0.73      0.92      0.81        24

    accuracy                           0.82        85
   macro avg       0.88      0.87      0.86        85
weighted avg       0.84      0.82      0.82        85

                 pred_ACTIVE  pred_NON_CODING  pred_INACTIVE  pred_GONE
true_ACTIVE                1                0              0          0
true_NON_CODING            0               28              0          5
true_INACTIVE              0                5             19          3
true_GONE                  0              

# Other
These next blocks are extra tools.
1st block checks for api key limits and prints them
2nd black is for the Truck factor program 

In [29]:
from github import Github

secrets = [

]
for new_token in secrets:
    ghub = Github(new_token)
    search_limit = ghub.get_rate_limit().search.remaining
    core_limit = ghub.get_rate_limit().core.remaining
    reset = ghub.get_rate_limit().core.reset
    #change the time to be in Mountain Standard Time (MST) 
    reset = reset.astimezone(tz=None)  # Convert to local timezone
    # Print the limits

    print(f"Search limit for token {new_token}:\n {search_limit}, {core_limit}, {reset} \n")


In [ ]:
#new
def findCoreDevelopers(
    url: str,
    dest_root: str | Path = ".tf_cache",
    *,
    name: str | None = None,
    branch: str | None = None,
    refresh: bool = False,
) -> tuple[list[str], list[str]]:        # <- correct annotation
    """
    Clone <url> (or reuse/refresh an existing clone) and run Truck‑Factor.
    Returns (authors, emails) – two parallel lists with the same length.
    """

    # ------------------------------------------------------------------ #
    # Paths
    # ------------------------------------------------------------------ #
    org, repo = url.rstrip("/").split("/")[-2:]
    repo = repo.removesuffix(".git")
    repo_path   = Path(cfg.main_folder) / org / repo          # actual repo
    cache_dir   = repo_path / dest_root                       # .tf_cache
    tf_csv      = cache_dir / "TruckFactor.csv"

    cache_dir.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------------ #
    # 1. Return cached result if possible
    # ------------------------------------------------------------------ #
    if tf_csv.is_file() and not refresh:
        cache_df = pandas.read_csv(
            tf_csv,
            sep=cfg.CSV_separator,
            encoding="utf-8",
        )
        return (
            cache_df["login"].tolist(),
            cache_df["email"].tolist(),
        )

    # ------------------------------------------------------------------ #
    # 2. Ensure we have a local clone
    # ------------------------------------------------------------------ #
    try:
        if (cache_dir / ".git").is_dir():
            repo = Repo(cache_dir)
            if refresh:
                repo.git.fetch("--all", "--prune")
            if branch:
                repo.git.checkout(branch)
                if refresh:
                    repo.git.pull()
        else:
            # Empty dir or non‑existent – clone afresh
            if cache_dir.exists():
                shutil.rmtree(cache_dir, ignore_errors=True)
            repo = Repo.clone_from(url, cache_dir, branch=branch)
    except git_exc.GitCommandError as e:
        raise RuntimeError(f"Git failed: {e.stderr or e}") from e

    # ------------------------------------------------------------------ #
    # 3. Compute Truck Factor (this calls your patched compute_tf)
    # ------------------------------------------------------------------ #
    tf, critical_sha, authors, emails = compute_tf(str(cache_dir))

    # Always lists from here on
    authors = list(authors)
    emails  = list(emails)

    # ------------------------------------------------------------------ #
    # 4. Cache the result for next time
    # ------------------------------------------------------------------ #
    pandas.DataFrame({"login": authors, "email": emails}).to_csv(
        tf_csv,
        sep=cfg.CSV_separator,
        index=False,
        lineterminator="\n",
        encoding="utf-8",
    )

    return authors, emails
#old
def findCoreDevelopers(
    url: str,
    dest_root: str | Path = ".tf_cache",
    *,
    name: str | None = None,
    branch: str | None = None,
    refresh: bool = False,
) -> tuple[int, str, list[str]]:
    """
    Clone <url> (or reuse/refresh an existing clone) and run Truck-Factor.
    Returns (tf, critical_sha, authors).
    """
    # --------------------------------------------------------------------- #
    dest_root = Path(dest_root).expanduser().resolve()    # .../rails/rails
    name = name or url.rstrip("/").split("/")[-1].removesuffix(".git")
    dest = dest_root / name
    tf_cache  = dest / ".tf_cache"
    tf_cache.mkdir(parents=True, exist_ok=True)                  
    tf_csv = dest / "TruckFactor.csv"
    
    clone_path = dest / name                          # .../.tf_cache/rails

    if tf_csv.is_file():
        logging.info("TF cache hit – using %s", tf_csv)
        return pandas.read_csv(tf_csv, encoding="utf-8")["login"].tolist()

    

    repo = None
    # --------------------------------------------------------------------- #
    try:
        if clone_path.exists():
            try:
                repo = Repo(clone_path)
            except git_exc.InvalidGitRepositoryError:
                # Directory exists but isn't a repo – start fresh
                shutil.rmtree(dest, ignore_errors=True)
                repo = Repo.clone_from(url, to_path=dest, branch=branch)
            else:
                # Repo is valid – refresh if asked
                if refresh:
                    repo.git.fetch("--all", "--prune")
                if branch:
                    repo.git.checkout(branch)
                    if refresh:
                        repo.git.pull()
        else:
            repo = Repo.clone_from(url, clone_path, branch=branch)
    except git_exc.GitCommandError as e:
        raise RuntimeError(f"Git failed: {e.stderr or e}") from e

    # --------------------------------------------------------------------- #
    # Ensure the repo is NOT empty (at least one commit reachable)
    if not list(repo.iter_commits('--all', max_count=1)):
        # Something went wrong – start over with a clean clone
        shutil.rmtree(dest, ignore_errors=True)
        repo = Repo.clone_from(url, clone_path, branch=branch)

    # --------------------------------------------------------------------- #
    # Truck-Factor
    #if the truck factor file does not exist, we compute it
    if not tf_csv.is_file():
        print("Computing Truck Factor for", clone_path)
        tf, critical_sha, authors = compute_tf(str(clone_path))
        
    else:
        print("Using cached Truck Factor from %s", tf_csv)
    

    pandas.DataFrame(authors, columns=["login"])\
      .to_csv(tf_csv ,
              sep=cfg.CSV_separator,
              index=False,
              lineterminator="\n",
              encoding="utf-8")

    return authors